In [2]:
# init
import importlib, sys

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from matplotlib.markers import MarkerStyle
from matplotlib.lines import Line2D

import superconductivity.api as sc

from superconductivity.api import G0_muS
from superconductivity.api import NDArray64

from IPython import get_ipython

from tqdm import tqdm

Textwidth: float = 4.25279  # in
Textheight: float = 6.85173  # in

_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("reload_ext", "autoreload")
    _ip.run_line_magic("autoreload", "2")

    _ip.run_line_magic(
        "config",
        "InlineBackend.print_figure_kwargs = {'bbox_inches': None, 'pad_inches': 0.0}",
    )
    _ip.run_line_magic("config", 'InlineBackend.figure_format = "retina"')  # or "png"
    _ip.run_line_magic("config", "InlineBackend.rc = {'figure.dpi': 300}")

# Simulation Parameter

In [4]:
GN_G0: float = 0.18877592218372993
Delta_meV: float = 0.19345000789195935
gamma_meV: float = 0.005066874981090785

Tbase_K = 0.0806
T0_K = 0.0

# Simulate Amplitude Study @ 18.3 GHz

In [3]:
Vbias0 = np.linspace(-6, 6, 2001)
Abias = np.linspace(0.0, 5.0, 101)

nu_GHz = 18.3

Vbias0_mV = Vbias0 * Delta_meV
Abias_mV = Abias * sc.h_pVs * nu_GHz

In [4]:
from superconductivity.utilities.meta.axis import axis
from superconductivity.utilities.meta.param import param

sim = sc.sim_bcs(
    V_mV=axis("V_mV", values=Vbias0_mV),
    GN_G0=param("GN_G0", GN_G0),
    T_K=param("T_K", T0_K),
    Delta_meV=param("Delta_meV", Delta_meV),
    gamma_meV=param("gamma_meV", gamma_meV),
    nu_GHz=param("nu_GHz", nu_GHz),
    A_mV=axis("A_mV", values=Abias_mV),
)
Isim0_nA = sim["I_nA"]
Isim0 = Isim0_nA / (GN_G0 * sc.G0_muS * Delta_meV)

In [5]:
Vbias = np.linspace(-0.5, 4.5, 151)
Ibias = np.linspace(-0.5, 4.5, 151)
Vbias_mV = Vbias * Delta_meV
Abias_mV = Abias * sc.h_pVs * nu_GHz
Ibias_nA = Ibias * GN_G0 * sc.G0_muS * Delta_meV

In [6]:
# binning
from superconductivity.utilities.functions.upsampling import upsample

Isim = sc.bin(
    z=Isim0,
    x=Vbias0,
    xbins=Vbias,
    axis=1,
)
dGsim = np.gradient(Isim, Vbias, axis=1)
dGsim = savgol_filter(dGsim, window_length=5, polyorder=0)

Vbias_up = upsample(Vbias0, N_up=100)
Isim_up = upsample(Isim0, N_up=100, axis=1)

Vsim = sc.bin(
    z=Vbias_up,
    x=Isim_up,
    xbins=Ibias,
    axis=1,
)
dRsim = np.gradient(Vsim, Ibias, axis=1)

In [7]:
# saving progress
np.savez_compressed(
    "amp_18.3GHz/sim.npz",
    Vbias=Vbias,
    Ibias=Ibias,
    Abias=Abias,
    Isim=Isim,
    dGsim=dGsim,
    dRsim=dRsim,
    nu_GHz=nu_GHz,
    GN_G0=GN_G0,
    Delta_meV=Delta_meV,
)

# Simulate Amplitude Study @ 13.6 GHz

In [8]:
Vbias0 = np.linspace(-6, 6, 2001)
Abias = np.linspace(0.0, 5.0, 101)

nu_GHz = 13.6

Vbias0_mV = Vbias0 * Delta_meV
Abias_mV = Abias * sc.h_pVs * nu_GHz

In [9]:
from superconductivity.utilities.meta.axis import axis
from superconductivity.utilities.meta.param import param

sim = sc.sim_bcs(
    V_mV=axis("V_mV", values=Vbias0_mV),
    GN_G0=param("GN_G0", GN_G0),
    T_K=param("T_K", T0_K),
    Delta_meV=param("Delta_meV", Delta_meV),
    gamma_meV=param("gamma_meV", gamma_meV),
    nu_GHz=param("nu_GHz", nu_GHz),
    A_mV=axis("A_mV", values=Abias_mV),
)
Isim0_nA = sim["I_nA"]
Isim0 = Isim0_nA / (GN_G0 * sc.G0_muS * Delta_meV)

In [10]:
Vbias = np.linspace(-0.5, 4.5, 151)
Ibias = np.linspace(-0.5, 4.5, 151)
Vbias_mV = Vbias * Delta_meV
Abias_mV = Abias * sc.h_pVs * nu_GHz
Ibias_nA = Ibias * GN_G0 * sc.G0_muS * Delta_meV

In [11]:
# binning
from superconductivity.utilities.functions.upsampling import upsample

Isim = sc.bin(
    z=Isim0,
    x=Vbias0,
    xbins=Vbias,
    axis=1,
)
dGsim = np.gradient(Isim, Vbias, axis=1)
dGsim = savgol_filter(dGsim, window_length=5, polyorder=0)

Vbias_up = upsample(Vbias0, N_up=100)
Isim_up = upsample(Isim0, N_up=100, axis=1)

Vsim = sc.bin(
    z=Vbias_up,
    x=Isim_up,
    xbins=Ibias,
    axis=1,
)
dRsim = np.gradient(Vsim, Ibias, axis=1)

In [12]:
# saving progress
np.savez_compressed(
    "amp_13.6GHz/sim.npz",
    Vbias=Vbias,
    Ibias=Ibias,
    Abias=Abias,
    Isim=Isim,
    dGsim=dGsim,
    dRsim=dRsim,
    nu_GHz=nu_GHz,
    GN_G0=GN_G0,
    Delta_meV=Delta_meV,
)

# Simulate Temperature Study

In [5]:
Vbias0 = np.linspace(-6, 6, 2001)
T_Tc = np.linspace(0.5, 1.0, 101)
Tc_K = sc.get_Tc_K(Delta_meV=Delta_meV)

Vbias0_mV = Vbias0 * Delta_meV
T_K = T_Tc * Tc_K

In [6]:
from superconductivity.utilities.meta.axis import axis
from superconductivity.utilities.meta.param import param

sim = sc.sim_bcs(
    V_mV=axis("V_mV", values=Vbias0_mV),
    GN_G0=param("GN_G0", GN_G0),
    T_K=axis("T_K", values=T_K),
    Delta_meV=param("Delta_meV", Delta_meV),
    gamma_meV=param("gamma_meV", gamma_meV),
)
Isim0_nA = sim["I_nA"]
Isim0 = Isim0_nA / (GN_G0 * sc.G0_muS * Delta_meV)

In [7]:
Vbias = np.linspace(-0.5, 4.5, 151)
Ibias = np.linspace(-0.5, 4.5, 151)
Vbias_mV = Vbias * Delta_meV
Ibias_nA = Ibias * GN_G0 * sc.G0_muS * Delta_meV

In [8]:
# binning
from superconductivity.utilities.functions.upsampling import upsample

Isim = sc.bin(
    z=Isim0,
    x=Vbias0,
    xbins=Vbias,
    axis=1,
)
dGsim = np.gradient(Isim, Vbias, axis=1)
dGsim = savgol_filter(dGsim, window_length=5, polyorder=0)

Vbias_up = upsample(Vbias0, N_up=100)
Isim_up = upsample(Isim0, N_up=100, axis=1)

Vsim = sc.bin(
    z=Vbias_up,
    x=Isim_up,
    xbins=Ibias,
    axis=1,
)
dRsim = np.gradient(Vsim, Ibias, axis=1)

In [11]:
# saving progress
np.savez_compressed(
    "temperature/sim.npz",
    Vbias=Vbias,
    Ibias=Ibias,
    Tsim=T_Tc,
    Isim=Isim,
    dGsim=dGsim,
    dRsim=dRsim,
    GN_G0=GN_G0,
    Delta_meV=Delta_meV,
)